# Analysis of Prot2Text-V2 + Foldseek predictions

Loads the artifacts written by the training notebook, builds one per-protein table across conditions, and interprets the results. Shared logic lives in `src/prot2text_foldseek/` (`analysis.py`, `metrics.py`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
!pip install -q "transformers>=4.48,<4.56" evaluate rouge_score bert_score sacrebleu

In [ ]:
import logging
import os
import sys

import pandas as pd

sys.path.insert(0, "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/src")

from prot2text_foldseek import analysis, metrics
from prot2text_foldseek.config import Config, DRIVE_ROOT

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4f}".format)

cfg = Config()
paths = cfg.paths
baseline_predict_path = os.path.join(DRIVE_ROOT, "Full_run", "test_baseline_predictions.pkl")
baseline_metric_path = os.path.join(DRIVE_ROOT, "multitask", "p2t_aux_checkpoints", "generations",
                                    "baseline_vs_finetuned_n4203.json")

## Load artifacts

In [ ]:
art = analysis.load_run_artifacts(paths)
baseline_predict, ext_matched, baseline_metrics = analysis.load_external_baseline(
    baseline_predict_path, baseline_metric_path)
print(f"external baseline: {len(baseline_predict):,} predictions, {len(ext_matched):,} matched accessions, "
      f"exact_match={baseline_metrics['exact_match']:.4f}")

## Per-protein master table

In [ ]:
df = analysis.build_master_frame(art, baseline_predict)
cond_order = [c for c in analysis.LOCAL_CONDITIONS if art.generations[c]]
print("DataFrame shape:", df.shape)
df.head(3)

## Aggregate metrics across conditions

In [ ]:
summary = metrics.comparison_table({
    "baseline": baseline_metrics,
    "seq_only": art.agg_metrics_seq,
    "without": art.agg_metrics.get("without_retrieval", {}),
    "with": art.agg_metrics.get("with_retrieval", {}),
})
summary

### Stratified by Foldseek identity bin (from the training run: with vs. without)

In [ ]:
delta_cols = [c for c in art.stratified.columns if c.startswith("d_")]

def colour_delta(v):
    if pd.isna(v):
        return ""
    return "color: green" if v > 0 else ("color: red" if v < 0 else "")

art.stratified.style.map(colour_delta, subset=delta_cols).format(
    {c: "{:.4f}" for c in art.stratified.select_dtypes(float).columns})

### Stratified: with retrieval vs. external baseline

In [ ]:
references = df["reference"].to_dict()
predictions = {
    "with": df["pred_with"].to_dict(),
    "baseline": {a: p for a, p in baseline_predict.items() if a in df.index},
}
bin_of = df["fident_bin"].to_dict()

strat_vs_baseline = metrics.stratified_metrics(references, predictions, bin_of,
                                               treatment="with", control="baseline")
strat_vs_baseline.to_csv(os.path.join(paths.out_dir, "stratified_baseline_vs_with.csv"), index=False)
strat_vs_baseline

## Inspecting individual proteins

In [ ]:
text_cols = ["reference", "best_hit_desc", "pred_baseline"] + [f"pred_{c}" for c in cond_order]
meta_cols = ["taxon", "fident_bin", "best_hit_fident", "best_hit_qcov", "best_hit_evalue",
             "best_hit_target", "best_hit_org"]

show_bin = ">50%"      # one of: "no hit", "<20%", "20-30%", "30-50%", ">50%"
df_bin = df[df["fident_bin"] == show_bin][meta_cols + text_cols]
print(f"{len(df_bin)} proteins in the {show_bin!r} bin")
df_bin.head(10)

In [ ]:
n_show = 5
subset = df[df["fident_bin"] == show_bin]
sample = subset.sample(min(n_show, len(subset)), random_state=42)

for acc, row in sample.iterrows():
    print("=" * 80)
    print(f"ACCESSION : {acc}")
    print(f"TAXON     : {row['taxon']}")
    if pd.notna(row["best_hit_fident"]):
        print(f"HIT       : {row['best_hit_target']}  {row['best_hit_fident'] * 100:.0f}% id  org={row['best_hit_org']}")
        print(f"HIT DESC  : {str(row['best_hit_desc'])[:200]}")
    print(f"\nREFERENCE : {row['reference']}\n")
    for cond in ["baseline"] + cond_order:
        pred = row[f"pred_{cond}"]
        if isinstance(pred, str):
            print(f"PRED ({cond:<14}): {pred}")
    print()

## Copying analysis

ROUGE-L between each prediction and the best retrieved description, per identity bin. High values in high-identity bins suggest the model copies the retrieved text. (Rebuilt from an earlier cell that referenced an undefined `copy_rows`; check this matches your original intent.)

In [ ]:
copy_df = analysis.copy_rate(df, [f"pred_{c}" for c in cond_order])
copy_summary = (copy_df.groupby("fident_bin").mean(numeric_only=True)
                .reindex([b for b in metrics.BIN_ORDER if b in copy_df["fident_bin"].unique()]).round(3))
copy_summary

## Per-protein metrics

In [ ]:
conditions = {c: df[f"pred_{c}"] for c in cond_order + ["baseline"]}

for cond, preds in conditions.items():
    valid = preds[preds.map(lambda p: isinstance(p, str))]       # baseline covers only a subset
    print(f"\nscoring {cond} ({len(valid):,} proteins)...")
    scores = metrics.per_protein_scores(valid.tolist(), df.loc[valid.index, "reference"].tolist(), cfg.metrics)
    for name, values in scores.items():
        df[f"{name}_{cond}"] = pd.Series(values, index=valid.index, dtype=float)

score_cols = [c for c in df.columns if c.split("_")[0] in ("rougeL", "bleu4", "bert")]
print("new score columns:", score_cols)

## Export

In [ ]:
export_path = os.path.join(paths.out_dir, "all_predictions_analysis.csv")
df.join(copy_df.drop(columns="fident_bin")).to_csv(export_path)
print(f"saved {len(df):,} rows -> {export_path}")